# AutoGen
AutoGen是微软开源的重量级 Multi-Agent 框架！作为多智能体领域的开山鼻祖级项目，AutoGen 凭借其优雅的 ConversableAgent（可对话智能体） 抽象，以及极其强大的 Code Execution（代码自动生成与本地环境真实执行） 机制，成为了自动化数据分析、代码重构与科学计算领域的首选框架。

## AutoGen 的 ConversableAgent 与代码执行闭环

1. 理解 AutoGen 的核心抽象：掌握 `ConversableAgent` 的底层通信机制，搞懂 `UserProxyAgent` 与 `AssistantAgent` 的角色定位。
2. 看透 AutoGen 如何自动识别 Markdown 中的代码块、在 Docker 或本地 Python 环境中安全运行并捕获终端输出。
3. 工程实战：搭建首个 AutoGen 全自动代码生成、运行与自纠错系统。


#### AutoGen 的两大核心角色哲学
在 AutoGen 的设计哲学中，一切 Agent 都是 `ConversableAgent` 的子类。它们通过互相发送消息（Messages）来推进任务。其中最经典、最常用的搭档是：
* AssistantAgent（AI 专家）：
    * 角色定位：充当“大脑”或“程序员”。
    * 核心能力：接收用户的自然语言需求，规划解决步骤，并输出包含 Python/Shell 代码块的回复，试图通过写代码来解决实际问题。

* UserProxyAgent（用户代理 / 执行器）：
    * 角色定位：充当“人类代表”或“代码执行终端”。
    * 核心能力：它通常不需要调用 LLM 大脑，而是默默监听 `AssistantAgent` 吐出的代码。一旦发现代码块，它会自动提取代码，在本地沙箱或终端中物理运行，并把运行结果（`stdout` 或报错信息 `stderr`）重新包装成消息塞回给 `AssistantAgent`！

#### 全自动“生成-执行-报错-修 Bug”闭环
AutoGen 最让人惊艳的地方，在于它把人类程序员“写代码 $\rightarrow$ 运行报错 $\rightarrow$ 看 Traceback 信息 $\rightarrow$ 修改代码”的研发循环，完全自动化了：
1. `UserProxyAgent` 发起需求：“帮我画一张 2024 年苹果与特斯拉股票走势图。”
2. AssistantAgent 编写代码：输出一段调用 `yfinance` 和 `matplotlib` 的 Python 代码。
3. `UserProxyAgent` 自动执行：
    * 自动在本地终端运行该代码。
    * 情况 A（报错）：终端报错 `ModuleNotFoundError: No module named 'yfinance'`。`UserProxyAgent` 自动捕捉这个报错，反馈给 `AssistantAgent`。
    * 情况 B（成功）：代码成功运行并保存了图片 `stock.png`，`UserProxyAgent` 输出执行成功信号。

4. `AssistantAgent` 自我修复：收到报错后，`AssistantAgent` 自动给出包含 `pip install yfinance` 或修复语法 Bug 的新代码，再次交给 `UserProxyAgent` 运行，直到成功为止。

#### 基于 AutoGen 构建数据分析自动化 Team
> 环境准备提示：运行本段代码需要安装 AutoGen 官方库：
`pip install autogen-agentchat flaml`

In [ ]:
import os
import autogen
from autogen import AssistantAgent, UserProxyAgent

# ================= 1. 配置 LLM 接入信息 =================
# AutoGen 统一使用 config_list 字典进行模型配置
config_list = [
    {
        "model": "gpt-4o",
        "api_key": os.getenv("OPENAI_API_KEY"),
    }
]

llm_config = {
    "config_list": config_list,
    "temperature": 0,
    "seed": 42, # 设定随机种子以保证结果可复现
}

# ================= 2. 创建 AssistantAgent (程序员) =================
assistant = AssistantAgent(
    name="Coder_Assistant",
    llm_config=llm_config,
    system_message="你是一名高水平的 Python 数据分析工程师。"
                   "遇到问题时，请编写完整的 Python 代码来解决。"
                   "当任务彻底完成、且代码执行无误后，请在回复的最后加上单词 TERMINATE。"
)

# ================= 3. 创建 UserProxyAgent (代码执行器) =================
user_proxy = UserProxyAgent(
    name="User_Proxy_Executor",
    human_input_mode="NEVER",  # 设定为 NEVER 表示完全自动化，不需要人类在终端按 Enter
    max_consecutive_auto_reply=8,  # 设置最大自动对话轮数，防止死循环
    is_termination_msg=lambda x: x.get("content", "").rstrip().endswith("TERMINATE"),
    code_execution_config={
        "work_dir": "autogen_workspace", # 代码执行与输出文件保存的根目录
        "use_docker": False, # 在本地环境或虚拟环境中直接运行 (生产环境强烈建议设为 True 使用 Docker 沙箱)
    }
)

# ================= 4. 启动 AutoGen 自动对话与代码执行循环 =================
task_prompt = """
请帮我写一段 Python 脚本：
1. 生成 100 个遵循正态分布的随机数。
2. 计算它们的均值（Mean）和标准差（Std）。
3. 使用 matplotlib 绘制出它们的直方图，并将图片保存为 'normal_distribution.png'。
4. 在终端打印出计算好的均值与标准差。
"""

print("🚀 启动 AutoGen 自动化代码编写与执行网络...\n")

# 由 user_proxy 启动与 assistant 的对话
user_proxy.initiate_chat(
    assistant,
    message=task_prompt
)

1. `human_input_mode` 的三种模式：
    * `"ALWAYS"`：每轮 Agent 发言后，系统都会在控制台暂停，等待人类输入指导意见（相当于微软版的 Human-in-the-Loop）。
    * `"NEVER"`：全自动模式，智能体完全自主写代码、执行代码并纠错。
    * `"TERMINATE"`：平时全自动，仅当收到终止信号（如 TERMINATE）或出错时才提示人类介入。

2. `code_execution_config` 的安全防护：
    * 在生产环境中，让 LLM 随意在服务器上执行代码具有极高的安全隐患（可能执行 `rm -rf` / 或泄漏环境变量）。因此，AutoGen 默认支持接入 Docker 容器，将代码隔离在沙箱内部物理执行。

3. `TERMINATE` 终止机制：
    在 `UserProxyAgent` 中通过 `is_termination_msg` 函数捕获特定字符串，这是控制 AutoGen 自动对话优雅退出的最稳妥方式。

**思考与练习**
1. 沙箱环境安全：如果你要在公司内部私有云上部署 AutoGen，除了将 `use_docker` 设为 `True` 之外，还可以采用哪些 Python 级别或操作系统级别的权限限制，以防止 LLM 生成的代码恶性访问敏感数据文件？
2. 多 Agent 扩展：查阅 AutoGen 官方文档，了解什么是 `GroupChat` 和 `GroupChatManager`。如果要把我们今天的单对单（`UserProxyAgent` + `AssistantAgent`）扩展成一个包含“产品经理、程序员、测试员”的群聊模式，AutoGen 是如何优雅实现多智能体群控的？